# Accessing LIBERO Dataset Samples

This notebook demonstrates how to access and explore LIBERO dataset samples.

The datasets are stored in HDF5 format. Each dataset contains multiple episodes (demonstrations) with:
- **Observations**: images (camera views), states (robot proprioception)
- **Actions**: 7-DOF robot actions
- **Metadata**: task descriptions, environment info


In [ ]:
%load_ext autoreload
%autoreload 2


# srun --time=4:0:0 --mem-per-cpu=32G --gpus=a100_80gb:1 --pty bash -l



import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TF logging
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'  # Disable oneDNN
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.95'
# To check if JAX is compiling functions
# os.environ['JAX_LOG_COMPILES'] = '1'  # Log when JAX compiles functions (cache hits/misses)


import h5py
import numpy as np

import json
from pathlib import Path
import gc
import jax

# Aggressively clear JAX caches and Python garbage at startup
jax.clear_caches()
gc.collect()

# Path to downloaded datasets (adjust if you downloaded to a different location)
# DATASET_DIR = "/cluster/scratch/anmari/libero_datasets"



In [ ]:
import jax
import gc
import sys
import jax.numpy as jnp

jax.clear_caches()


# Check current GPU memory allocation
def check_gpu_memory():
    # Get the primary GPU
    gpu = jax.devices("gpu")[0]
    stats = gpu.memory_stats()

    # Convert bytes to GB for readability
    used_gb = stats['bytes_in_use'] / 1e9
    reserved_gb = stats['pool_bytes'] / 1e9

    print(f"Actual model usage: {used_gb:.2f} GB")
    print(f"JAX Reserved (Pool): {reserved_gb:.2f} GB")

def clear_all_memory(delete_vars=None, auto_detect=True):
    """
    Aggressively clear all JAX/GPU memory after interrupting run_evaluation_ttt.

    Args:
        delete_vars: Optional list of variable names to delete. If None and auto_detect=True,
                    will try to find and delete common large objects.
        auto_detect: If True, automatically find and delete large objects (models, policies, etc.)
    """
    print("=" * 70)
    print("Clearing all memory...")
    print("=" * 70)

    # Check memory before
    print("\nMemory BEFORE clearing:")
    check_gpu_memory()

    # Get IPython namespace
    try:
        from IPython import get_ipython
        ipython = get_ipython()
        user_ns = ipython.user_ns if ipython is not None else globals()
    except:
        user_ns = globals()

    # Auto-detect large objects if requested
    if auto_detect and delete_vars is None:
        print("\n0. Auto-detecting large objects to delete...")
        large_objects = []
        common_names = ['policy_ttt', 'policy', 'model', 'trained_model', 'train_state',
                       'nn_fetcher', 'dataset', 'env', 'ttt_policy', 'model_copy',
                       'config', 'train_config', 'benchmark', 'suite']

        for name in common_names:
            if name in user_ns:
                obj = user_ns[name]
                try:
                    size = sys.getsizeof(obj)
                    if size > 1024:  # > 1KB
                        large_objects.append(name)
                        print(f"   Found: {name} (size: {size/1024:.2f} KB)")
                except:
                    large_objects.append(name)
                    print(f"   Found: {name} (could not get size)")

        if large_objects:
            delete_vars = large_objects
            print(f"   Will delete: {delete_vars}")
        else:
            print("   No common large objects found")

    # Delete specific variables if provided
    if delete_vars:
        print(f"\n1. Deleting specified variables: {delete_vars}")
        deleted = []
        not_found = []
        errors = []

        for var_name in delete_vars:
            try:
                if var_name in user_ns:
                    # Try to call .close() or cleanup methods if they exist
                    obj = user_ns[var_name]
                    if hasattr(obj, 'close'):
                        try:
                            obj.close()
                            print(f"   Called .close() on {var_name}")
                        except:
                            pass
                    if hasattr(obj, 'cleanup'):
                        try:
                            obj.cleanup()
                            print(f"   Called .cleanup() on {var_name}")
                        except:
                            pass

                    del user_ns[var_name]
                    deleted.append(var_name)
                else:
                    not_found.append(var_name)
            except Exception as e:
                errors.append((var_name, str(e)))

        if deleted:
            print(f"   ✓ Deleted: {deleted}")
        if not_found:
            print(f"   ✗ Not found: {not_found}")
        if errors:
            print(f"   ⚠ Errors: {errors}")
    else:
        print("\n1. No variables specified for deletion")

    # Clear JAX caches (compiled functions, etc.)
    print("\n2. Clearing JAX caches...")
    jax.clear_caches()

    # Try to clear XLA compilation cache more aggressively
    try:
        # Clear all devices
        for device in jax.devices():
            try:
                device.clear_cache()
            except:
                pass
    except:
        pass

    # Force garbage collection multiple times with different generations
    print("\n3. Running aggressive garbage collection...")
    for i in range(5):
        collected = gc.collect()
        if collected > 0:
            print(f"   GC pass {i+1}: collected {collected} objects")

    # Collect all generations
    for gen in range(3):
        collected = gc.collect(gen)
        if collected > 0:
            print(f"   GC generation {gen}: collected {collected} objects")

    # Try to clear JAX device memory by forcing synchronization
    print("\n4. Synchronizing JAX devices...")
    try:
        # Force all pending operations to complete
        for device in jax.devices():
            jax.block_until_ready(jax.device_put(0, device))
        print("   ✓ Devices synchronized")
    except Exception as e:
        print(f"   ⚠ Warning: Could not synchronize all devices: {e}")

    # Try to force JAX to release memory pool (this may not work, but worth trying)
    print("\n5. Attempting to release JAX memory pool...")
    try:
        # This is a hack - create and delete a large array to trigger memory management
        # JAX may release some memory when it sees low usage
        dummy = jax.device_put(jnp.zeros((100, 100), dtype=jnp.float32))
        jax.block_until_ready(dummy)
        del dummy
        gc.collect()
        print("   ✓ Triggered memory management")
    except Exception as e:
        print(f"   ⚠ Could not trigger memory management: {e}")

    # Final garbage collection
    print("\n6. Final garbage collection...")
    gc.collect()

    # Check memory after
    print("\nMemory AFTER clearing:")
    check_gpu_memory()

    # Important note about JAX memory pool
    gpu = jax.devices("gpu")[0]
    stats = gpu.memory_stats()
    pool_gb = stats.get('pool_bytes', 0) / 1e9
    if pool_gb > 1.0:
        print(f"\n⚠ NOTE: JAX memory pool still reserves {pool_gb:.2f} GB")
        print("   This is NORMAL - JAX keeps memory reserved for future allocations.")
        print("   The pool will shrink gradually as other processes need memory.")
        print("   To fully release memory, you may need to restart the kernel.")

    print("\n" + "=" * 70)
    print("Memory clearing complete!")
    print("=" * 70)

check_gpu_memory()

In [ ]:
clear_all_memory()
check_gpu_memory()

In [ ]:
import sys
_project_root = Path.cwd()
if not (_project_root / "meta_libero").exists():
    _project_root = _project_root.parent
sys.path.append(str(_project_root / "meta_libero"))
from libero_dataset import override_create_torch_dataset
from openpi.training import data_loader as _data_loader
import dataclasses
from openpi.training import config as _config

# Clear memory before loading model
import gc
import jax
# jax.clear_caches()
gc.collect()
print("Memory cleared before model loading")


# Test Nearest Neighbor Selection

This section tests the nearest neighbor fetcher to retrieve similar samples from the FAISS index.

In [ ]:
# Load liber0.5 model
from utils import load_pi05_libero_model
from pathlib import Path

model, config = load_pi05_libero_model(use_lora=True, action_expert_only=False)
check_gpu_memory()


In [ ]:
from nn_fetcher import NearestNeighborFetcher

DATASET_TO_USE = "libero_90"


# Path to FAISS index (adjust if needed)
if DATASET_TO_USE == "libero_10":
    REPO_ID = "physical-intelligence/libero"
    CACHE_DIR = Path.home() / ".cache" / "libero_unified_faiss"
else:
    assert DATASET_TO_USE == "libero_90"
    REPO_ID = "physical-intelligence/libero_90"
    CACHE_DIR = Path.home() / ".cache" / "libero_90_norm"

modality_str = "_".join(sorted(["image1", "image2", "text"]))
index_path = CACHE_DIR / f"libero_unified_faiss_index_{modality_str}.index"
metadata_path = CACHE_DIR / f"libero_unified_faiss_metadata_{modality_str}.pkl"

print(f"Index path: {index_path}")
print(f"Metadata path: {metadata_path}")
print(f"Index exists: {index_path.exists()}")
print(f"Metadata exists: {metadata_path.exists()}")
# Initialize fetcher
if index_path.exists() and metadata_path.exists():
    nn_fetcher = NearestNeighborFetcher(
        index_path=str(index_path),
        metadata_path=str(metadata_path),
        model=model,
    )
    print("✓ NearestNeighborFetcher initialized successfully!")
else:
    print("⚠ Index files not found. Please run build_unified_faiss_index.py first.")
    nn_fetcher = None

In [ ]:
# Get a sample observation from the dataloader
config = dataclasses.replace(config, batch_size=1)

with override_create_torch_dataset(repo_id=REPO_ID):  # /libero_90
    test_dataloader = _data_loader.create_data_loader(
        config,
        sharding=None,
        shuffle=False,
    )

# Get first batch
for batch in test_dataloader:
    observation, actions = batch
    break

print("Sample observation keys:", observation.images.keys() if hasattr(observation, 'images') else "N/A")
print("Tokenized prompt shape:", observation.tokenized_prompt.shape if hasattr(observation, 'tokenized_prompt') and observation.tokenized_prompt is not None else "N/A")

In [ ]:
# Import TTT evaluation function
from utils import create_policy

checkpoint_dir = "/cluster/home/anmari/.cache/openpi/openpi-assets/checkpoints/pi05_libero"

policy_ttt = create_policy(
    model,
    config,
    checkpoint_dir,
    rng_seed=42,  # For reproducibility
)
print("✓ Policy created for TTT evaluation")

In [ ]:
# Get the underlying dataset
ttt_dataset = test_dataloader._data_loader._data_loader.dataset
print(f"TTT dataset size: {len(ttt_dataset)} samples")


In [ ]:
nn_fetcher.index.ntotal

In [ ]:
jax.clear_caches()
check_gpu_memory()

In [ ]:
from importlib import reload
import utils
reload(utils)

In [ ]:
clear_all_memory(delete_vars=['trained_model', 'train_state', 'dataset', 'env', 'ttt_policy', 'model_copy'])

In [ ]:
from utils import run_evaluation_ttt


success_rate = run_evaluation_ttt(
    policy=policy_ttt,
    nn_fetcher=nn_fetcher,
    train_config=config,
    dataset=ttt_dataset,  # Dataset for indexing retrieved samples
    num_trials=1,  # Reduced for testing
    task_suite_name="libero_10",
    task_id=5, # was 8
    save_video=False,  # Set to True to save videos
    seed=42,
    ttt_num_steps=0,  # Number of gradient steps per TTT update
    ttt_frequency=20,  # Perform TTT every N steps during rollout
    learning_rate=2.5e-5,
    max_ttt_step=500, # was 150
    ttt_k=6,  # Number of nearest neighbors to retrieve
    random_neighbors=False,
    ttt_use_modalities=["image1", "image2", "text"],  # Modalities for retrieval
    plot_observations=True,
    reset_policy=False,
    repeat_batch=1,
    use_test_task=False,
    cfg_weight=1.0,
)

# Note: 2.5e-04 looks good for the first steps


In [ ]:
from utils import run_evaluation_noise


success_rate, all_episode_metrics = run_evaluation_noise(
    policy=policy_ttt,
    nn_fetcher=nn_fetcher,
    train_config=config,
    dataset=ttt_dataset,  # Dataset for indexing retrieved samples
    num_trials=1,  # Reduced for testing
    task_suite_name="libero_90",
    task_id=1, # was 8
    save_video=False,  # Set to True to save videos
    seed=42,
    noise_frequency=20,  # Perform TTT every N steps during rollout
    noise_sigma=0.1,
    max_noise_step=150,
    plot_observations=True,
    reset_policy=True,
)


# Note: 2.5e-04 looks good for the first steps

print(f"\n{'='*70}")
print(f"TTT Evaluation Complete!")
print(f"Success Rate: {success_rate*100:.1f}%")
print(f"{'='*70}")


# Other statistics (not related to previous cells)

In [ ]:
import numpy as np
from scipy import stats

def confidence_interval_95(data):
    """Compute 95% confidence interval for the mean."""
    data = np.array(data)
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)  # Standard error of the mean
    ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)
    return mean, ci[0], ci[1]

data = [0.24, 0.24, 0.22]
mean, ci_low, ci_high = confidence_interval_95(data)
print(f"No TTT - Mean: {mean:.4f}, 95% CI: [{ci_low:.4f}, {ci_high:.4f}]")